# PPE Training Pipeline — Colab Notebook

Trains **Model A** (our fused PPE detector), evaluates it against the hosted
generic **Model B**, and produces one zip with every number and figure the
paper needs. Full beginner walkthrough: `GUIDE.md` in the GitHub repository.

**Setup once:** Runtime → Change runtime type → **T4 GPU**. Create
`MyDrive/ppe-training/secrets.env`, then add a Colab Secret named
`GITHUB_TOKEN` with read access to the private repository (see `GUIDE.md`).

**Then just: Runtime → Run all.** Code is cloned fresh from GitHub; datasets,
checkpoints, and results remain on Drive.

Disconnected? Reconnect and **Run all** again. Finished steps skip themselves;
training resumes from the last epoch checkpoint on Drive. Worst case you lose
the epoch in progress.


## Setup — mount Google Drive

Everything important (datasets, the prepared fused dataset, every training
checkpoint, the final results zip) is written straight to your Drive under
`ppe-training/`, so nothing is lost when the Colab VM disappears. Approve the
access popup when it appears.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Setup — clone code from GitHub + install dependencies

Code lives in GitHub, not Drive. This cell clones the `main` branch fresh into
`/content/ppe-cv`, then installs `training/requirements.txt` on the Colab VM.
Drive is used only for secrets, datasets, checkpoints, and results.


In [ ]:
# Clone the private pipeline repo from GitHub; keep durable artifacts on Drive.
from google.colab import userdata
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import tempfile

REPO_URL = 'https://github.com/YasserDalali/ppe-cv.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/ppe-cv')
CODE = Path('/content/ppe-cv/training')
DRIVE_ROOT = Path('/content/drive/MyDrive/ppe-training')
REQS = CODE / 'requirements.txt'

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'Add a Colab secret named GITHUB_TOKEN with read access to YasserDalali/ppe-cv.'
    ) from exc
github_token = (github_token or '').strip()
if not github_token:
    raise RuntimeError('Colab secret GITHUB_TOKEN is empty.')

# Prefer GIT_ASKPASS over Bearer headers / token URLs: more reliable for
# fine-grained PATs and keeps the token out of argv / clone logs.
askpass = Path(tempfile.mkdtemp(prefix='git-askpass-')) / 'askpass.sh'
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) printf %s "x-access-token" ;;\n'
    '  *) printf %s "$GIT_PASSWORD" ;;\n'
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_PASSWORD': github_token,
})
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(REPO_DIR, ignore_errors=True)
clone = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    env=git_env, capture_output=True, text=True,
)
askpass.unlink(missing_ok=True)
askpass.parent.rmdir()
del github_token, git_env
if clone.returncode != 0:
    err = (clone.stderr or clone.stdout or '').strip()
    raise RuntimeError(
        'git clone failed. Check: (1) Colab secret GITHUB_TOKEN has Notebook access, '
        '(2) fine-grained token includes repo YasserDalali/ppe-cv with Contents: Read-only, '
        f'(3) branch {REPO_BRANCH} exists.\nGit said:\n{err}'
    )

if not REQS.is_file():
    raise FileNotFoundError(f'GitHub clone is missing {REQS}')
subprocess.run(['pip', 'install', '-q', '-r', str(REQS)], check=True)
if importlib.util.find_spec('ultralytics') is None:
    raise RuntimeError(f'Failed to install deps from {REQS}')
print(f'code ready from {REPO_URL}@{REPO_BRANCH}')
print('deps ready (Model B uses REST if inference package absent)')


using existing code at /content/drive/MyDrive/ppe-training/training
offline install incomplete — falling back to PyPI (refreshing wheelhouse)
deps ready (Model B uses REST if inference package absent)


## Configuration — the only cell you might edit

All fill-in variables and hyperparameters live in one `Config` object
(YOLOv8n, 640×640, batch 16, 50 epochs, seed 0, the dataset slugs, the
×3 duplication factors...). Defaults match the paper spec, so usually you
edit **nothing**.

Secrets are read from `MyDrive/ppe-training/secrets.env`
(`ROBOFLOW_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY`) — never put keys in
this notebook or in code. If the file is missing, this cell tells you
exactly where to create it.


In [ ]:
# CONFIG — the only cell you might edit. Secrets come from
# <DRIVE_ROOT>/secrets.env (ROBOFLOW_API_KEY / KAGGLE_USERNAME / KAGGLE_KEY).
import sys
sys.path.insert(0, str(CODE / 'src'))
from pathlib import Path
from ppe.config import CANONICAL_CLASSES, Config, load_secrets

cfg = Config(drive_root=DRIVE_ROOT, work_root=Path('/content/ppe-work'))
secrets = load_secrets(cfg)
print('config OK —', len(CANONICAL_CLASSES), 'classes:', CANONICAL_CLASSES)


## Step 1 — Download the 4 datasets (skipped if already prepared)

If `prepared/fused.zip` is already on Drive (built locally and uploaded —
see `training/docs/LOCAL_PREPARE.md`), this step is skipped entirely: no
Roboflow/Kaggle calls, no raw datasets land on Drive. Otherwise it fetches
into `Drive/ppe-training/raw/` (each source is downloaded **once, ever** — a
`.complete` marker makes re-runs skip it):

1. the public Roboflow **PPE** set (also the data Model B was trained on),
2. **SH17** from Kaggle (industrial PPE classes),
3. the **gas-masks** set (v1, 384 images — fixed, don't substitute),
4. the **OCP** site photos (Roboflow project `OCP`).

OCP is required — if its download fails the run stops here with instructions
(most likely fix: generate **Version 1** of the OCP project in the Roboflow
UI, plain YOLOv8 export).


In [ ]:
# STEP 1 — download & cache all 4 dataset sources (skips finished ones).
# Skipped entirely when prepared/fused.zip is already on Drive (dataset was
# built locally and uploaded) — ensure_prepared() in the next cell only
# needs the raw sources to build fused.zip in the first place.
if (cfg.prepared_dir / 'fused.zip').is_file():
    print('[download] prepared/fused.zip already on Drive — skipping raw download')
    raw_paths = {}
else:
    from ppe.download import download_all
    raw_paths = download_all(cfg, secrets)
raw_paths


## Steps 2 + 3 — Remap labels, then merge + split

**Remap:** every label file is rewritten onto our single 10-class list
(`Person, helmet, vest, gloves, boots, goggles, no_helmet, no_gloves,
no_goggle, gas mask`) by **class name**, never by raw id. Anything that can't
be mapped stops the run loudly — no silent data loss (the old run lost ~43%
of images exactly here). The mapping + drop accounting is saved as
`remap.json`.

**Merge + split:** one fused dataset, split 70/15/15 train/val/test with a
fixed seed (identical split every run). OCP and gas-mask train images are
physically duplicated ×3 so the model sees them more often. Two held-out
eval sets are built: **ocp_test** (OCP images never trained on) and
**industrial_proxy** (SH17 + gas-mask test images, no OCP). Exact counts land
in `split_summary.json`.

The result is zipped to Drive (`prepared/fused.zip`, built **once**) and
restored to the VM's fast local disk on every run.

**Already prepared locally?** `ensure_prepared()` below checks
`prepared/fused.zip` first — if it's already on Drive (uploaded after running
`scripts/prepare_local.py`, see `training/docs/LOCAL_PREPARE.md`) it skips
rebuilding entirely and just restores it to fast local disk.


In [ ]:
# STEPS 2+3 — remap by name (hard-fail accounting) + merge/split/holdouts.
# Builds once, zips to Drive, restores to fast local disk on every run.
from ppe.merge import ensure_prepared
local_dataset = ensure_prepared(cfg, secrets)
data_yaml = local_dataset / 'data.yaml'
print('dataset ready at', local_dataset)


## Step 4 — Train Model A (ours)

YOLOv8n at 640×640, batch 16, 50 epochs, HSV + flip + erase + mosaic
augmentation (mosaic off for the last 10 epochs), on the Colab T4. Expect a
few hours.

Checkpoints (`last.pt` / `best.pt`) are written to **Drive after every
epoch**, so a disconnect costs at most the epoch in progress: on the next
*Run all* this cell notices the unfinished run and resumes from `last.pt`
automatically; a finished run is skipped entirely.


In [ ]:
# STEP 4 — train Model A (auto-resumes; checkpoints on Drive every epoch)
from ppe.train import train_model_a
best_pt = train_model_a(cfg, data_yaml)
best_pt


## Step 5 — Model B (generic) — no training

Model B is the **existing hosted Roboflow model** `ppe_detection-dnfen/1`
(the only version with a trained endpoint; published P/R/mAP50 match v1).
We never train it and no `generic.pt` exists — we only run it over our test
images. Preferred backend is the `inference` package (the model runs locally
in this VM, no per-image API metering); if that's unavailable it falls back
to the hosted REST API. Its class names are read at runtime and matched to
ours case-insensitively.


In [ ]:
# STEP 5 — build both predictors (Model B = hosted Roboflow model, never trained)
from ppe.predict import RoboflowPredictor, UltralyticsPredictor
predictor_ours = UltralyticsPredictor(best_pt, conf=cfg.eval_conf, iou=cfg.eval_iou, imgsz=cfg.imgsz)
# Model B metadata is always the hosted DatasetPPE model (v1), even if the
# training download was pointed at a fork after the Universe CDN broke.
_gm_proj, _gm_ver = cfg.generic_model_id.split("/", 1)
predictor_generic = RoboflowPredictor(
    cfg.generic_model_id, secrets['ROBOFLOW_API_KEY'],
    conf=cfg.eval_conf, iou=cfg.eval_iou,
    metadata_coords=("datasetppe", _gm_proj, int(_gm_ver)))
print('Model B backend:', predictor_generic.backend)


## Step 6 — Evaluate both models with ONE shared harness

The exact same scoring code runs both models over the same images with the
same confidence/IoU settings, restricted to the **shared classes** (person,
helmet, vest, gloves + the no_* variants — intersected with what Model B
actually knows). That fills the 6-row comparison table:

| Model | Eval set |
|---|---|
| Generic | Own val (published numbers, copied as-is) |
| Generic | Industrial proxy / OCP site |
| Ours | Fused val / Industrial proxy / OCP site |

Model A additionally gets the Ultralytics per-class P/R/mAP50 table, speed
numbers (→ "~N cameras at X FPS"), and the normalized confusion matrix.


In [3]:
# STEP 6 — one shared harness for both models + Model A extras
from ppe.evaluate import compare_all, model_a_val
eval_sets = {name: cfg.work_root / name
             for name in ('fused_val', 'industrial_proxy', 'ocp_test')}
comparison = compare_all(cfg, predictor_ours, predictor_generic, eval_sets)
extras = model_a_val(cfg, best_pt, data_yaml)
comparison['rows']


ModuleNotFoundError: No module named 'ppe'

## Resources — Compare Model A and B on the same images

In [2]:
import random
from pathlib import Path

# Check if Model A (ours) is trained by verifying 'best_pt' exists and points to a file.
# 'best_pt' is expected to be a Path object from the training step.
if 'best_pt' not in locals() or not isinstance(best_pt, Path) or not best_pt.is_file():
    print("Model A (ours) is not trained or 'best_pt' is not available. Skipping inference comparison.")
else:
    # Get validation image paths from the local dataset
    # This part of the code assumes 'local_dataset', 'predictor_ours', and 'predictor_generic'
    # are defined and available from previously executed cells.
    val_images_dir = local_dataset / 'images' / 'val'
    image_paths = list(val_images_dir.glob('*.jpg')) + list(val_images_dir.glob('*.png'))

    if not image_paths:
        print(f"No images found in {val_images_dir}. Skipping inference comparison.")
    else:
        # Select up to 30 random images for comparison
        num_inferences = min(30, len(image_paths))
        selected_images = random.sample(image_paths, num_inferences)

        print(f"Running {num_inferences} inferences for comparison:")

        for i, img_path in enumerate(selected_images):
            print(f"\n--- Image {i+1}/{num_inferences}: {img_path.name} ---")

            # Model A (Our model) prediction
            try:
                preds_ours = predictor_ours.predict(img_path)
                print(f"Model A (Ours) detections ({len(preds_ours)}):")
                # Print top 3 detections if any, for brevity
                for j, det in enumerate(preds_ours[:3]):
                    # Assuming 'class_name', 'confidence', 'box' are keys in detection dict
                    print(f"  {j+1}. Class: {det.get('class_name', 'N/A')}, Conf: {det.get('confidence', 0):.2f}, Box: {det.get('box', 'N/A')}")
                if len(preds_ours) > 3:
                    print(f"  ...and {len(preds_ours) - 3} more.")
            except Exception as e:
                print(f"Error running Model A prediction for {img_path.name}: {e}")

            # Model B (Generic model) prediction
            try:
                preds_generic = predictor_generic.predict(img_path)
                print(f"Model B (Generic) detections ({len(preds_generic)}):")
                # Print top 3 detections if any, for brevity
                for j, det in enumerate(preds_generic[:3]):
                    # Assuming 'class_name', 'confidence', 'box' are keys in detection dict
                    print(f"  {j+1}. Class: {det.get('class_name', 'N/A')}, Conf: {det.get('confidence', 0):.2f}, Box: {det.get('box', 'N/A')}")
                if len(preds_generic) > 3:
                    print(f"  ...and {len(preds_generic) - 3} more.")
            except Exception as e:
                print(f"Error running Model B prediction for {img_path.name}: {e}")


NameError: name 'local_dataset' is not defined

## Report — build the handoff zip (the deliverable)

Writes `results/handoff-<date>/` and zips it: `best.pt`, `remap.json`,
`data.yaml`, a fully-filled `metrics.md` (generation **fails** if any blank
survives), `results.csv`, `split_summary.json`, the training dashboard +
confusion-matrix figures, and a `README.md` answering the 5 required
questions (gas-masks v1 confirmed; val vs test; epochs + GPU; low-image
classes; alert system "designed only" — plus: Model B hosted → no
`generic.pt`; Step 7 / LLM judge skipped).

Send the authors the single file `results/handoff-<date>.zip`. Done.


In [ ]:
# REPORT — metrics.md (zero blanks, enforced), README answers, handoff zip
import json, torch, ultralytics
from ppe.report import build_handoff, build_metrics_md, build_readme, low_image_classes

run_dir = cfg.runs_dir / cfg.run_name
summary = json.loads((cfg.work_root / 'split_summary.json').read_text())
training = {
    'epochs': cfg.epochs,
    'ultralytics_version': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'headline_split': 'val',
}
metrics_md = build_metrics_md(comparison, extras, training)
readme = build_readme(cfg, {
    # remap.json (inside the prepared dataset) has this even when Step 1
    # was skipped and raw/ was never downloaded onto this Drive.
    'gasmask_images': json.loads((cfg.work_root / 'remap.json').read_text())
        ['reports']['gasmask']['images_before'],
    'headline_split': 'val',
    'epochs': cfg.epochs,
    'gpu': training['gpu'],
    'low_image_classes': low_image_classes(summary),
})
handoff = build_handoff(cfg, {
    'best_pt': best_pt,
    'remap_json': cfg.work_root / 'remap.json',
    'data_yaml': data_yaml,
    'results_csv': run_dir / 'results.csv',
    'split_summary': cfg.work_root / 'split_summary.json',
    'dashboard_png': run_dir / 'results.png',
    'confusion_png': extras['confusion_matrix_png'],
    'metrics_md': metrics_md,
    'readme_md': readme,
})
print('handoff folder:', handoff)
print('zip written next to it in', cfg.results_dir)
